# 07 — Five training seeds, one fixed split
Run after notebook 05. Reuse the saved `samples.pt` and its exact sequence IDs/class weights. Labels are unchanged.

Only initialization, dropout **masks**, and whole-sequence sampling vary with the seed. The dropout **rate**, architecture, learning rate, epochs, batch size, class weights and sampling weight stay fixed. Each seed selects its best epoch by macro F1, matching notebook 05; ties keep the earlier epoch.

Completed matching seed runs are reused. Interrupted seeds restart from epoch 1. Original `runs/best.pt` and its logs are preserved. Report all five seeds: mean, sample standard deviation (`ddof=1`), minimum, maximum and range. These describe training variability on this validation split, not uncertainty across unseen sequences.


In [1]:
# Cell purpose: Freeze the existing split/settings and configure five independent training seeds.
# Inputs: samples.pt and the original baseline checkpoint settings when available.
# Outputs: Five seeds and one shared parameter dictionary; no split regeneration or training here.

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from config import CONFIG, fingerprint, save_json, require_cache
from training_runs import train_one_seed, summarize_runs, split_manifest, validate_split

# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
print('Sample cache:', CACHE / 'samples.pt')
# Existing samples are authoritative: never rerun splitting for a training seed.
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'samples.pt', '04_build_training_samples.ipynb')
data = torch.load(CACHE / 'samples.pt', map_location='cpu', weights_only=True)
# Check disjoint sequence/page groups, class support and original training-only label counts.
validate_split(data)
original = RUN / 'best.pt'
if original.exists():
    checkpoint = torch.load(original, map_location='cpu', weights_only=True)
    # Reject a checkpoint whose sequence/features/split cache differs from the current samples.
    assert checkpoint['samples_signature'] == data['signature'], 'Sample cache changed since baseline; use its matching cache.'
    baseline_config = checkpoint.get('config', CONFIG)
else:
    baseline_config = CONFIG
# Keep the original training settings fixed; the seed experiment changes randomness rather than hyperparameters.
BASE_PARAMS = {
    'lr': float(baseline_config['lr']), 'weight_decay': 0.01, 'dropout': 0.1,
    'batch_size': int(baseline_config['batch_size']), 'epochs': int(baseline_config['epochs']),
    'narration_sampling_weight': float(baseline_config['narration_sampling_weight']),
}
SPLIT = split_manifest(data)
print('Fixed split:', len(SPLIT['train_sequences']), 'train /', len(SPLIT['val_sequences']), 'validation sequences')
print('Original train class counts:', SPLIT['train_counts'])
print('Fixed training settings:', BASE_PARAMS)

# CONFIG: choose five different seeds; use a new RUN_NAME when changing settings.
# Each seed controls initialization, dropout masks and sequence sampling; the saved split stays unchanged.
SEEDS = [42, 123, 2026, 7, 99]
RUN_NAME = 'five_seeds'
REUSE_COMPLETED = True
OUTPUT = RUN / RUN_NAME
assert len(SEEDS) == 5 and len(set(SEEDS)) == 5

Sample cache: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\cache\samples.pt
Fixed split: 64 train / 16 validation sequences
Original train class counts: [1142, 50]
Fixed training settings: {'lr': 0.0003, 'weight_decay': 0.01, 'dropout': 0.1, 'batch_size': 4, 'epochs': 20, 'narration_sampling_weight': 2.0}


In [2]:
# Cell purpose: Train or reuse each seed run and summarize all five best-epoch results.
# Inputs: One fixed split/settings manifest and five different training random seeds.
# Outputs: Separate seed artifacts plus mean/std/min/max/range for narration P/R/F1 and macro F1.

OUTPUT.mkdir(parents=True, exist_ok=True)
# Record the exact experiment identity and reject incompatible results on a later rerun.
manifest = {'split': SPLIT, 'params': BASE_PARAMS, 'seeds': SEEDS, 'selection': 'macro_f1'}
manifest_file = OUTPUT / 'manifest.json'
if manifest_file.exists():
    assert json.loads(manifest_file.read_text(encoding='utf-8')) == manifest, 'Settings changed; choose a new RUN_NAME.'
save_json(manifest_file, manifest)

results = []
# Run each seed independently; train_one_seed reuses only a completed matching run.
for seed in SEEDS:
    result = train_one_seed(data, BASE_PARAMS, seed, OUTPUT / f'seed_{seed}', reuse=REUSE_COMPLETED)
    results.append(result)
    save_json(OUTPUT / 'results.json', results)

# Report every seed plus mean, sample std, minimum, maximum and range; do not choose the luckiest seed.
per_seed, summary = summarize_runs(results, OUTPUT)
display(per_seed)
display(summary)
print('Saved:', OUTPUT / 'per_seed_metrics.csv', 'and', OUTPUT / 'seed_summary.csv')

seed 42 epoch 01 | train 0.2701 | val 0.1406 | narration F1 0.000 | macro F1 0.488
seed 42 epoch 02 | train 0.1569 | val 0.1247 | narration F1 0.105 | macro F1 0.538
seed 42 epoch 03 | train 0.1074 | val 0.1188 | narration F1 0.111 | macro F1 0.542
seed 42 epoch 04 | train 0.0474 | val 0.2668 | narration F1 0.000 | macro F1 0.489
seed 42 epoch 05 | train 0.0642 | val 0.3486 | narration F1 0.000 | macro F1 0.488
seed 42 epoch 06 | train 0.0309 | val 0.1940 | narration F1 0.235 | macro F1 0.606
seed 42 epoch 07 | train 0.0102 | val 0.0937 | narration F1 0.593 | macro F1 0.787
seed 42 epoch 08 | train 0.0052 | val 0.2143 | narration F1 0.333 | macro F1 0.656
seed 42 epoch 09 | train 0.0016 | val 0.3361 | narration F1 0.421 | macro F1 0.701
seed 42 epoch 10 | train 0.0039 | val 0.2003 | narration F1 0.455 | macro F1 0.717
seed 42 epoch 11 | train 0.0097 | val 0.1380 | narration F1 0.667 | macro F1 0.825
seed 42 epoch 12 | train 0.0010 | val 0.1832 | narration F1 0.571 | macro F1 0.778
seed

,seed,best_epoch,val_loss,narration_precision,narration_recall,narration_f1,macro_f1
0,42,11,0.137969,0.642857,0.692308,0.666667,0.825425
1,123,15,0.165304,0.666667,0.461538,0.545455,0.764016
2,2026,7,0.081515,0.444444,0.615385,0.516129,0.744790
3,7,18,0.167082,0.470588,0.615385,0.533333,0.754299
4,99,9,0.113405,0.615385,0.615385,0.615385,0.798920


,mean,std,min,max,range
narration_precision,0.567988,0.102882,0.444444,0.666667,0.222222
narration_recall,0.600000,0.084265,0.461538,0.692308,0.230769
narration_f1,0.575394,0.063458,0.516129,0.666667,0.150538
macro_f1,0.777490,0.033711,0.744790,0.825425,0.080635


Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\per_seed_metrics.csv and C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\seed_summary.csv


In [3]:
# Cell purpose: Show where each seed prediction CSV lives and confirm the shared validation sequence IDs.
# Inputs: The completed five-seed experiment and its fixed split manifest.
# Outputs: Printed file paths and validation IDs for manual comparison.

# Each best checkpoint also has its own complete per-box validation CSV.
# Run each seed independently; train_one_seed reuses only a completed matching run.
for seed in SEEDS:
    print(seed, OUTPUT / f'seed_{seed}' / 'validation_predictions.csv')
print('Same validation IDs for every run:', SPLIT['val_sequences'])

42 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\seed_42\validation_predictions.csv
123 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\seed_123\validation_predictions.csv
2026 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\seed_2026\validation_predictions.csv
7 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\seed_7\validation_predictions.csv
99 C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTraining\runs\five_seeds\seed_99\validation_predictions.csv
Same validation IDs for every run: ['seq_72fdfff4b4353a42', 'seq_1c1d4f5e71a2f554', 'seq_f7ee32f1dd417802', 'seq_210721b0d576149f', 'seq_fa2eee6269a5dad3', 'seq_d749b853cf795b6a', 'seq_0dc2e3d874c804fc', 'seq_de542b63e6b19bc2', 'seq_f9043534f1171d4e', 'seq_84541a1f4b159018', 'seq_f9248